# <center> library </center>

In [1]:
import pandas as pd
import numpy as np
import time
import tracemalloc
from psutil import Process

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout, Attention, GlobalAveragePooling1D

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

# <center> dataframe </center>

In [ ]:
input_path = f'your-path\\IDS-IoT\\dataset\\TON-IoT\\normal-attack\\csv-6\\ton.csv'
ton_df = pd.read_csv(input_path, on_bad_lines="error", low_memory=False)

# <center> train/validation </center>

In [ ]:
X = ton_df.drop(['label', 'type'], axis=1).values
y = ton_df['label'].values
X = X.reshape((X.shape[0], 1, X.shape[1]))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def HARN(input_shape):
    inputs = Input(shape=input_shape)
    lstm_out = LSTM(128, return_sequences=True)(inputs)
    gru_out = GRU(128, return_sequences=True)(lstm_out)
    attention_layer = Attention()([lstm_out, gru_out])
    attention_output = GlobalAveragePooling1D()(attention_layer)
    dense_output = Dense(64, activation='relu')(attention_output)
    dropout_output = Dropout(0.1)(dense_output)
    outputs = Dense(1, activation='sigmoid')(dropout_output)
    model = Model(inputs=inputs, outputs=outputs)
    return model

input_shape = (X_train.shape[1], X_train.shape[2])
model = HARN(input_shape)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.TruePositives(name='tp'),
        tf.keras.metrics.TrueNegatives(name='tn'),
        tf.keras.metrics.FalsePositives(name='fp'),
        tf.keras.metrics.FalseNegatives(name='fn')
    ]
)

tracemalloc.start()

start_time = time.time()
history = model.fit(X_train, y_train, epochs=70, batch_size=64, validation_split=0.2)
end_time = time.time()
current, peak = tracemalloc.get_traced_memory()
training_time = end_time - start_time
scalability = Process().memory_info().rss / (1024 ** 2)

tracemalloc.stop()

# <center> train/validation performance summary <center>

In [ ]:
print("📊 Training Performance Summary")
print(f"⏱ Training Time:        {training_time:.2f} seconds")
print(f"📦 Current RAM Usage:   {current / (1024 ** 2):.2f} MB")
print(f"🚀 Peak RAM Usage:      {peak / (1024 ** 2):.2f} MB")
print(f"📈 Scalability (RSS):   {scalability:.2f} MB")

# <center> train/validation model summary <center>

In [ ]:
model.summary()

# <center> train/validation charts <center>

## <center> accuracy, precision, recall, auc, tp, tn, fp, and fn <center>

In [ ]:
metrics_to_plot = [
    'accuracy', 'precision', 'recall', 'auc',
    'tp', 'tn', 'fp', 'fn'
]

for metric in metrics_to_plot:
    plt.figure(figsize=(8, 4))
    plt.plot(history.history[metric], label=f'Train {metric}')
    plt.plot(history.history[f'val_{metric}'], label=f'Validation {metric}')
    plt.xlabel('Epoch')
    plt.ylabel(metric.upper())
    plt.title(f'Training vs Validation {metric.upper()}')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## <center> loss <center>

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Binary Crossentropy Loss')
plt.legend()
plt.grid(True)
plt.show()

## <center> f1 <center>

In [ ]:
train_precision = history.history['precision']
train_recall = history.history['recall']
val_precision = history.history['val_precision']
val_recall = history.history['val_recall']

train_f1 = [2 * (p * r) / (p + r) for p, r in zip(train_precision, train_recall)]
val_f1 = [2 * (p * r) / (p + r) for p, r in zip(val_precision, val_recall)]

plt.figure(figsize=(8, 4))
plt.plot(train_f1, label='Training F1-score')
plt.plot(val_f1, label='Validation F1-score')
plt.title('F1-score Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True)
plt.show()


## <center> specifity and fpr <center>

In [13]:
tp = history.history['tp']
tn = history.history['tn']
fp = history.history['fp']
fn = history.history['fn']

val_tp = history.history['val_tp']
val_tn = history.history['val_tn']
val_fp = history.history['val_fp']
val_fn = history.history['val_fn']

In [14]:
train_specificity = [tn[i] / (tn[i] + fp[i] + 1e-8) for i in range(len(tn))]
val_specificity = [val_tn[i] / (val_tn[i] + val_fp[i] + 1e-8) for i in range(len(val_tn))]

train_fpr = [fp[i] / (fp[i] + tn[i] + 1e-8) for i in range(len(fp))]
val_fpr = [val_fp[i] / (val_fp[i] + val_tn[i] + 1e-8) for i in range(len(val_fp))]


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_specificity, label='Training Specificity')
plt.plot(val_specificity, label='Validation Specificity')
plt.title('Specificity Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Specificity')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_fpr, label='Training FPR')
plt.plot(val_fpr, label='Validation FPR')
plt.title('False Positive Rate (FPR) Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('FPR')
plt.legend()
plt.grid(True)
plt.show()


# <center> AUC-ROC test chart <center>

In [ ]:
y_pred_prob = model.predict(X_test).ravel()

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label="AUC-ROC Curve (area = %0.2f)" % roc_auc)
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("AUC-ROC Curve")
plt.legend(loc="lower right")
plt.show()

# <center> test <center>

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype("int32")
y_test_binary = y_test.astype("int32")

results = model.evaluate(X_test, y_test, verbose=0)

f1 = f1_score(y_test_binary, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test_binary, y_pred).ravel()
specificity = tn / (tn + fp)
fpr = fp / (fp + tn)

print(f"F1 Score:                  {f1:.4f}")
print(f"Specificity:              {specificity:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print(f"Loss (Binary Crossentropy): {results[0]:.4f}")
print(f"Accuracy:                  {results[1]:.4f}")
print(f"Precision:                 {results[2]:.4f}")
print(f"Recall:                    {results[3]:.4f}")
print(f"AUC:                       {results[4]:.4f}")
print(f"True Positives:            {results[5]}")
print(f"True Negatives:            {results[6]}")
print(f"False Positives:           {results[7]}")
print(f"False Negatives:           {results[8]}")


# <center> confusion matrix test <center>

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=['Normal', 'Attack'],
    cmap='Blues',
    normalize=None
)

plt.title('unnormalized Confusion Matrix')
plt.tight_layout()
plt.show()

# <center> t-sne <center>

In [ ]:
embedding_model = Model(inputs=model.input,
                        outputs=model.get_layer("dense").output)

X_embeddings = embedding_model.predict(X_test)

tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', random_state=42)
X_tsne = tsne.fit_transform(X_embeddings)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_test, cmap='coolwarm', alpha=0.6)
plt.title("t-SNE Projection of HARN Embeddings")
plt.xlabel("TSNE Dimension 1")
plt.ylabel("TSNE Dimension 2")
plt.colorbar(scatter, label='Class (0 = Normal, 1 = Attack)')
plt.grid(True)
plt.tight_layout()
plt.show()
